In [ ]:
#!pip install piragi
#!pip install langfuse
from piragi import Ragi
import os
import time
import json
from datetime import datetime
#from langfuse import Langfuse
from typing import Optional, List, Dict, Any
import hashlib
import gc

#langfuse = Langfuse()
CONFIG = {
    "llm": {
        "model": "qwen3.5-35b-32k", #контекстное окно (32k токенов)
        "base_url": "https://llm.ai-expert-opinion.ru/v1",
        "api_key": os.getenv("LLM_API_KEY", "sk-bc9a3a95e190fc9cf5525a4726b16658"),
        "max_tokens": 4096, ## макс. длина ответа модели
        "temperature": 0.01,  #чем ниже, тем меньше креатива
    },
    "embedding": {
        "model": "sentence-transformers/paraphrase-MiniLM-L3-v2",
        "batch_size": 32,
    },
    "chunk": {
        "strategy": "semantic", #разбиение на чанки по смыслу
        "size": 800,
        "overlap": 100,  #перекрытие чанков, чтобы не потерять контекст на стыках
    },
    "retrieval": {
        "use_hybrid_search": True, #гибридный поиск: BM25 (ключевые слова) + эмбеддинги (смысл)
        #bm25 (tf-idf range по совп слов) + reranking (присваиваем документам с пом эмб ранки по смыслу)
        "use_cross_encoder": False, # для запроса и эмб док разные энкодеры
        "priority_documents": ["kardio-25-04-493.pdf", "recommendation.pdf"],
        "min_relevance_score": 0.3,  #порог отсечения слабых совпадений
        "max_chunks_per_doc": 3 #лимит чанков на один документ в выдаче
    },
    "cache": {
        "enabled": False, #True=do caching
        "ttl_hours": 24,  #хранение ответов в часах
    }}

def clear_memory():
    gc.collect() #запуск сборщика мусора из неиспользуемых объектов
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512' #оптимизация выделения памяти CUDA
    print("Память очищена")

#генерация ключа кэша
def _get_cache_key(query: str, json_data: Dict) -> str:
    content = f"{query}|{json.dumps(json_data, sort_keys=True)}"
    return hashlib.md5(content.encode()).hexdigest()

def _load_from_cache(key: str, cache_dir: str = ".rag_cache") -> Optional[Dict]:
    if not CONFIG["cache"]["enabled"]:
        return None
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"{key}.json")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

def _save_to_cache(key: str, result: Dict, cache_dir: str = ".rag_cache"):
    #key: str — хеш-ключ, result: Dict — словарь с ответом LLM для сохранения
    if not CONFIG["cache"]["enabled"]:
        return
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"{key}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

#Поиск с приоритизацией
def retrieve_with_priority(kb: Ragi, query: str, priority_docs: List[str], top_k: int = 3, min_score: float = 0.3) -> Dict[str, Any]:
    #kb: Ragi - экземпляр базы знаний (индексированные документы), top_k - сколько чанков вернуть в итоге
    #min_score - минимальный порог релевантности
    citations = kb.retrieve(query, top_k=30)
    chunks = [c for c in citations if c.score >= min_score]
    result = []
    for doc in priority_docs:
        doc_chunks = [c for c in chunks if doc.lower() in c.source.lower()]
        result.extend(doc_chunks)
        if len(result) >= top_k:
            break
    result.sort(key=lambda c: c.score, reverse=True)
    return {"chunks": result[:top_k],
            "sources": list(set(c.source for c in result[:top_k])) #уникальные источники
           }

#prompt engineering
def format_prompt(json_data: Dict[str, Any], retrieved_context: str, instruction: str = "Сформируйте заключение врача-кардиохирурга, основываясь на данных измерений КТ аорты") -> str:
    json_formatted = json.dumps(json_data, ensure_ascii=False, indent=2) #Dict->json
    return f"""<system>
Вы — врач-кардиохирург, эксперт в области кардиологии и сосудистой хирургии с пятидесятилетним опытом работы.
Ваша задача: сформировать профессиональное медицинское заключение на основе предоставленных данных.

ПРАВИЛА:
1. Анализируйте данные последовательно, нужен предельно качественный ответ
2. Указывайте только подтверждённые измерениями факты, если данных нет, указывайте "данные не предоставлены"
3. Используйте необходимую медицинскую терминологию
4. Структура ответа: [Анализ данных] → [Интерпретация] → [Диагноз+Рекомендации]
</system>

<patient_data>
{json_formatted}
</patient_data>

<clinical_context>
{retrieved_context}
</clinical_context>

<task>
{instruction}

Пожалуйста, рассуждайте по шагам:
1. Какие ключевые параметры аорты представлены в данных?
2. Есть ли отклонения от референсных значений?
3. Какой клинический вывод следует из сопоставления измерений и контекста?
4. Сформулируйте итоговое заключение: диагноз и рекомендации врача.
</task>"""

#do not remember include this function in code
#def truncate_context(context: str, max_tokens: int = 2000) -> str:
#    return context[:max_tokens] if len(context) > max_tokens else context

def ask_with_rag(query: str, json_data: Dict[str, Any], kb: Ragi, top_k: int = 5) -> Dict[str, Any]:
    #сразу проверяем есть ли этот запрос + json в кеше, если есть повторно не используем llm
    cache_key = _get_cache_key(query, json_data)
    cached = _load_from_cache(cache_key)
    if cached:
        print("Ответ загружен из кэша")
        return cached

    #trace = langfuse.trace(name="rag_medical_query", user_id="mashaa",
    #    session_id=f"session_{int(time.time())}", metadata={"model": CONFIG["llm"]["model"], "top_k": top_k})
    try:
        retrieval_result = retrieve_with_priority(kb=kb, query=query, priority_docs=CONFIG["retrieval"]["priority_documents"],
                                                  top_k=top_k, min_score=CONFIG["retrieval"]["min_relevance_score"])
        if not retrieval_result["chunks"]:
            print("Контекст не найден ни в одном из документов")
            context_text = "Дополнительный клинический контекст не предоставлен."
        else:
            context_text = "\n\n".join([
                f"[{i + 1}] {c.chunk} (источник: {c.source}, релевантность: {c.score:.2%})"
                for i, c in enumerate(retrieval_result["chunks"])
            ])
            avg_score = sum(c.score for c in retrieval_result["chunks"]) / len(retrieval_result["chunks"])
            print(f"Средняя релевантность: {avg_score:.2%}")

        prompt = format_prompt(json_data=json_data, retrieved_context=context_text, instruction=query)

        answer = kb.ask(prompt, top_k=1)
        result = {
            "query": query,
            "answer": answer.text,
            "timestamp": datetime.now().isoformat(),
            "citations": [
                {"source": c.source, "score": c.score, "chunk": c.chunk}
                for c in retrieval_result["chunks"]
            ],
            "sources_used": retrieval_result["sources"]
        }

        filename = f"rag_response_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        _save_to_cache(cache_key, result)
        print(f"Ответ сохранён в: {filename}")

        #trace.generation( name="rag_output",
        #    output={"answer": answer.text, "citations_count": len(result["citations"])},
        #    usage_details={"total_citations": len(result["citations"])})
        #trace.update(output={"status": "success"})

        return result

    except Exception as e:
        #trace.update(output={"status": "error", "error": str(e)})
        raise
    #finally:
        #langfuse.flush()

def is_index_exist(persist_dir: str) -> bool:
    if not os.path.exists(persist_dir):
        return False
    if not os.listdir(persist_dir):
        return False
    return True

def initialize_kb(docs_path: str = "/content/docs",
                  persist_dir: str = "/content/.piragi_index"):
    clear_memory()
    if is_index_exist(persist_dir):
        print("Найден индекс, загружаем его")
    else:
        print("Индекс не найден, создаём новый")
    kb = Ragi(docs_path, persist_dir=persist_dir, config=CONFIG)
    return kb

start_time = time.time()
kb = initialize_kb()

with open(f"/content/0.json", "r", encoding="utf-8") as f:
    patient_data=json.load(f)

result = ask_with_rag(query="Сформируйте заключение врача-кардиохирурга, основываясь на данных измерений КТ аорты", json_data=patient_data, kb=kb, top_k=3)
print(f"\nЗаключение:\n{result['answer']}")
print(f"\nВремя выполнения: {time.time() - start_time:.2f} сек")


Память очищена
Найден индекс, загружаем его


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Контекст не найден ни в одном из документов
Ответ сохранён в: rag_response_20260325_171335.json

Заключение:


# Медицинское заключение врача-кардиохирурга

## [Анализ данных]

**1. Ключевые параметры аорты в предоставленных данных:**

Согласно предоставленным данным измерений КТ аорты, представлены следующие сегменты:
- Нисходящая аорта (Descending Aorta): max_diam_1 = 29.15 мм, max_diam_2 = 28.65 мм, min_diam = 25.50 мм, площадь = 580 мм²
- Истмус (Isthmus): max_diam_1 = 33.62 мм, max_diam_2 = 32.65 мм, min_diam = 26.88 мм, площадь = 705 мм²
- Дуга после левой подключичной артерии (Arch after LSA): max_diam_1 = 29.73 мм, max_diam_2 = 29.41 мм, min_diam = 22.77 мм, площадь = 533 мм²
- Дуга после плечеголовного ствола (Arch after TBC): max_diam_1 = 29.73 мм, max_diam_2 = 29.41 мм, min_diam = 25.18 мм, площадь = 577 мм²
- Восходящая аорта до плечеголовного ствола (Ascending Aorta before TBC): max_diam_1 = 39.45 мм, max_diam_2 = 38.68 мм, min_diam = 35.01 мм, площадь = 1040 мм²
- Восходя